In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.api as sm

from sklearn.linear_model import LassoCV, ridge_regression
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_absolute_error

In [3]:
def read_data(prefix: str, max_hops: int) -> pd.DataFrame:
    dfs = []
    for i in range(3, max_hops + 1):
        df = pd.read_json(f'../results/{prefix}_{i}.json')
        df['source'] = i
        df['hops'] = i - 1
        df['true_size_bytes'] = df['size_bytes'] * (i - 1)

        dfs.append(df)

    df = pd.concat(dfs)

    # Remove outliers
    df = df[(df["true_size_bytes"] > 64) | (df["time_s"] < 0.01)]

    return df



nccl_df = read_data('quest/nccl', max_hops=8)
gloo_df = read_data('quest/gloo', max_hops=8)
orin_df = read_data('orin/all_gather', max_hops=4)

In [4]:
def display_plot(df: pd.DataFrame):
    # Aggregate data by mean time_s for each size_bytes and source
    df_agg = df.groupby(['true_size_bytes', 'source'])['time_s'].mean().reset_index()

    # Create the plot
    fig = px.line(df_agg, x='true_size_bytes', y='time_s', color='source', title='Bytes over Time (Mean)')
    fig.show()

def display_data_points(df: pd.DataFrame, means: bool = False):
    if means:
        df = df.groupby(['true_size_bytes', 'source'])['time_s'].mean().reset_index()
    df2 = df.copy()
    df2['source'] = df2['source'].astype(str)  # make categorical so colors are discrete

    # build a stable color map for sources
    unique_sources = sorted(
        df2['source'].unique(),
        key=lambda x: (int(x) if str(x).isdigit() else x)
    )
    palette = px.colors.qualitative.Plotly
    color_map = {s: palette[i % len(palette)] for i, s in enumerate(unique_sources)}

    # scatter plot of data points with discrete colors
    fig = px.scatter(
        df2,
        x='true_size_bytes',
        y='time_s',
        color='source',
        # title='Bytes over Time (Data Points)',
        color_discrete_map=color_map
    )

    # guiding lines: mean latency per (true_size_bytes, source)
    df_means = df.groupby(['true_size_bytes', 'source'])['time_s'].mean().reset_index()
    df_means['source'] = df_means['source'].astype(str)

    import plotly.graph_objects as go
    for s in unique_sources:
        d = df_means[df_means['source'] == s].sort_values('true_size_bytes')
        if d.empty:
            continue
        fig.add_trace(
            go.Scatter(
                x=d['true_size_bytes'],
                y=d['time_s'],
                mode='lines',
                line=dict(color=color_map[s], width=2),
                name=f'{s} mean',
                opacity=0.25,
                showlegend=False
            )
        )

    fig.update_layout(width=500, height=500)
    # Zoom into the origin (0,0) by a factor of 5: show the first 1/5 of each axis range
    x_max = df2['true_size_bytes'].max()
    y_max = df2['time_s'].max()
    fig.update_xaxes(range=[0, x_max / 4])
    fig.update_yaxes(range=[0, y_max / 4])
    try:
        # Requires the 'kaleido' package: pip install -U kaleido
        fig.write_image("gloo_time_vs_bytes.pdf", format="pdf", engine="kaleido")
        print("Saved plot to gloo_time_vs_bytes.pdf")
    except Exception as e:
        print("Failed to save PDF (is 'kaleido' installed?). Falling back to interactive display. Error:", e)
    fig.show()

display_data_points(gloo_df, means=False)

display_plot(nccl_df)
display_plot(gloo_df)
display_plot(orin_df)

/var/folders/ym/j7xrj8pj0vlfstnbbgqq1vt00000gn/T/ipykernel_20613/600494169.py:62: DeprecationWarning:


Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.




Saved plot to gloo_time_vs_bytes.pdf


In [5]:
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_percentage_error

def evaluate(y_true, y_pred, X, n_bins=8):
    y_true, y_pred, X = np.asarray(y_true), np.asarray(y_pred), np.asarray(X)
    if X.ndim == 1:
        X = X.reshape(-1, 1)

    # --- Basic metrics ---
    print(f"R² = {r2_score(y_true, y_pred):.4f}")
    print(f"MAPE = {mean_absolute_percentage_error(y_true, y_pred):.4f}")
    smape = np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_pred) + np.abs(y_true)))
    print(f"SMAPE = {smape:.4f}")

    # --- Variance normalization (bin-based) ---
    bins = [np.linspace(X[:, i].min(), X[:, i].max(), n_bins + 1) for i in range(X.shape[1])]
    idx = [np.digitize(X[:, i], bins[i]) - 1 for i in range(X.shape[1])]
    sigma = np.zeros_like(y_true)

    # group by bins (1D or 2D)
    for b1 in range(n_bins):
        for b2 in ([0] if X.shape[1] == 1 else range(n_bins)):
            mask = (idx[0] == b1) & (idx[-1] == b2)
            if np.sum(mask) > 1:
                sigma[mask] = np.std(y_true[mask] - y_pred[mask])
    sigma[sigma == 0] = np.median(sigma[sigma > 0])
    norm_mae = np.mean(np.abs(y_true - y_pred) / sigma)
    print(f"Norm. MAE = {norm_mae:.4f}")

    # --- MacroMAPE (uniform over X space) ---
    mapes = []
    for b1 in range(n_bins):
        for b2 in ([0] if X.shape[1] == 1 else range(n_bins)):
            mask = (idx[0] == b1) & (idx[-1] == b2)
            if np.sum(mask) > 0:
                mapes.append(mean_absolute_percentage_error(y_true[mask], y_pred[mask]))
    macro_mape = np.mean(mapes)
    print(f"MacroMAPE = {macro_mape:.4f}")


In [8]:

from sklearn.discriminant_analysis import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline


def fit_model(df: pd.DataFrame):
    df = df.copy()
    # df["interaction"] = df["true_size_bytes"] * df["hops"]

    df_agg = df.groupby(["true_size_bytes", "hops"])["time_s"].mean().reset_index()
    df_agg = df_agg[df_agg["time_s"] < 10]

    X = df_agg[["true_size_bytes", "hops"]]
    y = df_agg["time_s"]

    # model = Pipeline([
    #     # ('scaler', StandardScaler()),
    #     ('model', LassoCV(cv=5, positive=True, fit_intercept=False))  # drop positive=True unless justified
    # ]).fit(X, y)
    # # model = LassoCV(cv=5, positive=True, fit_intercept=False).fit(X, y)

    # mm = model.named_steps["model"]
    # coefs = pd.Series(mm.coef_, index=X.columns)
    # print("Coefficients:")
    # for name, val in coefs.items():
    #     print(f"  {name} = {val:.6e}")
    # if hasattr(mm, "intercept_"):
    #     print(f"Intercept = {mm.intercept_:.6e}")

    model = LinearRegression()
    smallest_loc = df_agg["true_size_bytes"] == df_agg["true_size_bytes"].min()
    a = (y[smallest_loc] / X[smallest_loc]["hops"]).mean()
    print(f"Anticipated latency per transmit (a): {a*1000:.4f} ms")
    bigger_loc = df_agg["true_size_bytes"] >= df_agg["true_size_bytes"] .max() * 0.1
    b = (y[bigger_loc] / X[bigger_loc]["true_size_bytes"]).mean()
    print(f"Anticipated inverse bandwidth (b): {1/(b * 1e6):.4f} MB/s")
    model.coef_ = np.array([b, a])
    model.feature_names_in_ = np.array(["true_size_bytes", "hops"])
    model.intercept_ = 0

    y_pred = model.predict(X)

    evaluate(y, y_pred, X)

    # Plot actual vs predicted
    fig = px.scatter(
        x=y, y=y_pred, labels={"x": "Actual Time (s)", "y": "Predicted Time (s)"}, # title="Actual vs Predicted Time"
    )
    fig.add_shape(type="line", x0=y.min(), y0=y.min(), x1=y.max(), y1=y.max(), line=dict(color="Red", dash="dash"))
    fig.show()

    # Save figure as pdf
    fig.update_layout(
        width=80 * 5,
        height=90 * 5,
        margin=dict(l=8, r=8, t=16, b=8),  # reduce padding around the plot
        autosize=False,
        legend=dict(tracegroupgap=0)
    )
    fig.write_image("orin_actual_vs_predicted.pdf", format="pdf", engine="kaleido")
    print("Saved plot to orin_actual_vs_predicted.pdf")

print("NCCL Model:")
# fit_model(nccl_df)
print("Gloo Model:")
# fit_model(gloo_df)
print("Orin Model:")
fit_model(orin_df)

NCCL Model:
Gloo Model:
Orin Model:
Anticipated latency per transmit (a): 0.4794 ms
Anticipated inverse bandwidth (b): 100.2427 MB/s
R² = 1.0000
MAPE = 0.0424
SMAPE = 0.0415
Norm. MAE = 3.1024
MacroMAPE = 0.0091


/var/folders/ym/j7xrj8pj0vlfstnbbgqq1vt00000gn/T/ipykernel_20613/174161711.py:60: DeprecationWarning:


Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.




Saved plot to orin_actual_vs_predicted.pdf


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/dogac/code/edge-llm-benchmark/.venv/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
    ~~~~~~~~^^
  File "/Users/dogac/code/edge-llm-benchmark/.venv/lib/python3.13/site-packages/ipykernel/kernelbase.py", line 302, in dispatch_control
    await self.process_control(msg)
  File "/Users/dogac/code/edge-llm-benchmark/.venv/lib/python3.13/site-packages/ipykernel/kernelbase.py", line 308, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/Users/dogac/code/edge-llm-benchmark/.venv/lib/python3.13/site-packages/jupyter_client/session.py", line 994, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/